In [2]:
import torch
from torch_geometric.nn import GraphConv, to_hetero
import torch 

class GCN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super(GCN, self).__init__()
        self.conv = GraphConv(-1, hidden_channels)

    def forward(self, x, edge_index):

        x = self.conv(x, edge_index)
        return x

model = GCN( hidden_channels=5 )

In [3]:
from torch_geometric.data import HeteroData
from torch_geometric.utils import to_networkx


r"""
User(Bob) 0          User(Tom) 1 
          \         /       ↖
           \       /         \ 
            \     /           \
             ➘   ↙             ➘   
              Image <--------User (Dan) 2 
                0 
        (3 in homo graph)
"""

data                = HeteroData()
num_users           = 3
num_user_features   = 2
num_posts           = 1

data['user'].x  = torch.randn( num_users, num_user_features, dtype=torch.float )
data['image'].x = torch.empty( (1, 0),  dtype=torch.float )

data['user', 'follows', 'user'].edge_index = torch.tensor( [ [1], 
                                                             [2] ], dtype=torch.long )

data['user', 'post', 'image'].edge_index   = torch.tensor( [ [0], 
                                                             [0] ], dtype=torch.long )

data['user', 'likes', 'image'].edge_index  = torch.tensor( [ [1,2], 
                                                             [0,0] ], dtype=torch.long )

assert data.validate() is True
assert data.has_isolated_nodes() is False
assert data.has_self_loops() is False

homogenous_data = data.to_homogeneous()
graph = to_networkx(homogenous_data)
print(f"Graph edges are {graph.edges}") # 3 is the image 


Graph edges are [(0, 3), (1, 2), (1, 3), (2, 3)]


In [4]:
model = to_hetero(model, data.metadata())
print(f"Model is \n{model}")

out = model(data.x_dict, data.edge_index_dict)
print(f"Output is \n{out}")

print(f"\n\nWeights of the MODEL")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model is 
GraphModule(
  (conv): ModuleDict(
    (user__follows__user): GraphConv(-1, 5)
    (user__post__image): GraphConv(-1, 5)
    (user__likes__image): GraphConv(-1, 5)
  )
)



def forward(self, x, edge_index):
    x_dict = torch_geometric_nn_to_hetero_transformer_get_dict(x);  x = None
    x__user = x_dict.get('user', None)
    x__image = x_dict.get('image', None);  x_dict = None
    edge_index_dict = torch_geometric_nn_to_hetero_transformer_get_dict(edge_index);  edge_index = None
    edge_index__user__follows__user = edge_index_dict.get(('user', 'follows', 'user'), None)
    edge_index__user__post__image = edge_index_dict.get(('user', 'post', 'image'), None)
    edge_index__user__likes__image = edge_index_dict.get(('user', 'likes', 'image'), None);  edge_index_dict = None
    conv__user = self.conv.user__follows__user(x__user, edge_index__user__follows__user);  edge_index__user__follows__user = None
    conv__image1 = self.conv.user__post__image((x__user, x__image), edge_index

In [11]:
from torch_geometric.nn import HeteroConv

class HeteroGNN(torch.nn.Module): 
    def __init__(self, hidden_channels):
        super(HeteroGNN, self).__init__()
        self.conv1 = HeteroConv({
            ('user', 'follows', 'user'): GCN(hidden_channels),
            ('user', 'post', 'image'): GCN(hidden_channels),
            ('user', 'likes', 'image'): GCN(hidden_channels),
        })

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv1(x_dict, edge_index_dict)
        return x_dict
    
model = HeteroGNN(hidden_channels=5)
print(model)
output = model(data.x_dict, data.edge_index_dict)
print(output)

HeteroGNN(
  (conv1): HeteroConv(num_relations=3)
)
{'user': tensor([[ 1.6008,  0.0522,  0.4806,  0.0584, -1.3610],
        [-0.0221,  0.1921, -0.1712,  0.0189,  0.3244],
        [-0.4038,  0.2235,  0.1066, -0.3133, -0.1313]], grad_fn=<AddBackward0>), 'image': tensor([[-0.7269, -0.1859,  0.9533, -0.4923,  1.7363]], grad_fn=<SumBackward1>)}


In [5]:
import torch 

embedding = torch.nn.Embedding(10, 3)
print(embedding.weight)


Parameter containing:
tensor([[ 0.4715,  0.8691,  0.3648],
        [ 0.7909, -0.5877,  0.1574],
        [ 0.6496, -0.6394,  0.2683],
        [-0.1191, -0.3943, -1.0016],
        [-0.0469, -0.7004, -1.1778],
        [ 0.6255, -0.0790,  1.5845],
        [ 0.7723,  0.1969, -0.3299],
        [ 0.9445, -0.2248, -0.7579],
        [ 0.3833, -0.2143, -0.0116],
        [ 0.4171,  0.4362,  0.3163]], requires_grad=True)
